In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Emotion model
emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"
emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_name)
emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_name)
emotion_model.eval().to(device)


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
           

In [3]:
@torch.no_grad()
def get_emotion_vector(text):
    inputs = emotion_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    logits = emotion_model(**inputs).logits
    probs = F.softmax(logits, dim=1)
    return probs.squeeze(0).detach().cpu()  # shape: [6]


In [4]:
import torch
import torch.nn as nn
from transformers import RobertaModel, RobertaTokenizer
import pandas as pd

class RobertaCNNWithEmotion(nn.Module):
    def __init__(self, roberta_model='roberta-base', num_classes=2, dropout=0.5):
        super(RobertaCNNWithEmotion, self).__init__()
        self.roberta = RobertaModel.from_pretrained(roberta_model)
        self.conv1 = nn.Conv1d(in_channels=768, out_channels=100, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(100 + 7, num_classes)  # +6 for emotion vector

    def forward(self, input_ids, attention_mask, emotion_vec):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state  # (B, L, 768)
        x = x.permute(0, 2, 1)         # (B, 768, L)
        x = self.conv1(x)              # (B, 100, L)
        x = self.relu(x)
        x = torch.max(x, dim=2).values # (B, 100)
        x = self.dropout(x)
        x = torch.cat([x, emotion_vec], dim=1)  # Concatenate emotion vector
        logits = self.fc(x)
        return logits


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        emotion_vec = get_emotion_vector(text)  # [6]

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'emotion_vec': emotion_vec,
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [6]:
from tqdm import tqdm

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct = 0, 0
    loop = tqdm(dataloader, desc="Training", leave=False)

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        emotion_vec = batch['emotion_vec'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, emotion_vec=emotion_vec)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()

        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

    acc = total_correct / len(dataloader.dataset)
    return total_loss / len(dataloader), acc
    
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_correct = 0, 0
    all_preds, all_labels = [], []

    loop = tqdm(dataloader, desc="Validating", leave=False)

    with torch.no_grad():
        for batch in loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            total_correct += (preds == labels).sum().item()

            loop.set_postfix(loss=loss.item())

    acc = total_correct / len(dataloader.dataset)
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


In [7]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_f1 = 0
        self.counter = 0
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, f1, model):
        if f1 > self.best_f1 + self.delta:
            self.best_f1 = f1
            self.counter = 0
            self.best_model_state = model.state_dict()
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


In [8]:
def load_combined_dataset(path=r'E:\Cyberbullying\dataset\11\combined_dataset.csv'):
    """ """
    df=pd.read_csv(path)
    print(f"Loaded combined dataset from {path}")
    print("Combined Label distribution:\n", df['label'].value_counts()) #
    return df[['text', 'label']]

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
from transformers import get_scheduler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

df = load_combined_dataset()
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, stratify=df['label']
)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(class_weights)

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
train_dataset = EmotionDataset(train_texts, train_labels, tokenizer)
val_dataset = EmotionDataset(val_texts, val_labels, tokenizer)
print("data tokenization")
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
print("data loades")
model = RobertaCNNWithEmotion().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5,weight_decay=0.01)


# Compute class weights

# Apply to loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 10
early_stopper = EarlyStopping(patience=3, delta=0.001)
# Scheduler
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=EPOCHS * len(train_loader),
)
train_losses, train_accuracies = [], []
val_losses, val_accuracies, val_f1s = [], [], []
print("training")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_f1s.append(val_f1)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train     | Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
    print(f"  Validation| Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f} | F1 Score: {val_f1:.4f}")

    early_stopper(val_f1, model)
    if early_stopper.early_stop:
        print("Early stopping triggered.")
        break
    scheduler.step()
# Load best model weights after training
model.load_state_dict(early_stopper.best_model_state)


Loaded combined dataset from E:\Cyberbullying\dataset\11\combined_dataset.csv
Combined Label distribution:
 label
1    118807
0    118778
Name: count, dtype: int64


NameError: name 'VidgenDataset' is not defined

In [ ]:
def plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies, val_f1s):
    epochs = range(1, len(train_losses)+1)
    plt.figure(figsize=(16,5))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs, train_accuracies, label='Train Acc')
    plt.plot(epochs, val_accuracies, label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs, val_f1s, label='Val F1', color='purple')
    plt.title('F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies, val_f1s)


NameError: name 'plot_metrics' is not defined

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def final_evaluation(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Not Hate', 'Hate']))

    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Hate', 'Hate'], yticklabels=['Not Hate', 'Hate'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()


In [ ]:
final_evaluation(model, val_loader, device)


NameError: name 'model' is not defined